# T2.1 – DBRepo Schema Setup
**Vienna Weather Wet-Month Prediction Experiment**

This notebook creates the database and tables in DBRepo via the Python REST client,
and adds descriptive metadata to make the database citable.

**Source dataset:** Stadt Wien. *Wetter seit 1872 Hohe Warte Wien*. data.gv.at, CC BY 4.0.  
**Original publisher:** Stadt Wien / MA 23 (https://www.data.gv.at)  
**License:** CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/)  
**Dataset URL:** https://www.data.gv.at/datasets/69a06550-1ede-4f50-9c36-e7fb5cf6e7e8

## 0. Install & import dependencies

In [3]:
# Install the DBRepo Python client (match version shown in bottom-left of the DBRepo UI)
!pip install dbrepo==1.13.4 pandas requests --quiet


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: C:\Users\azras\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import requests
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    QueryDefinition,
    FilterDefinition,
    FilterType,
    OrderDefinition,
    OrderType,
)
import pandas as pd
import time



## 1. Configuration
Fill in your DBRepo credentials and the container ID before running.
The container ID can be found in the DBRepo UI under *Admin → Containers*.

In [4]:
# ── EDIT THESE ──────────────────────────────────────────────
ENDPOINT  = "https://test.dbrepo.tuwien.ac.at"
USERNAME  = "azra1558"   
PASSWORD  = "Katalizator1558!"   

DATABASE_NAME    = "vienna_weather_wet_months"
CSV_URL          = "https://www.wien.gv.at/data/ogd/ma23/vie-bdl-ecl-wea-1872f.csv"
client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)


In [7]:
print("Current user:", client.whoami())


azra1558
Current user: azra1558


## 2. Connect to DBRepo

In [8]:

# Verify connection by listing existing databases
dbs = client.get_databases()
print(f"Connected. Found {len(dbs)} existing database(s).")

Connected. Found 5 existing database(s).


## 3. Create the database

In [16]:
containers = client.get_containers()
print(containers)
CONTAINER_ID = containers[0].id

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [17]:
db = client.create_database(
    container_id=CONTAINER_ID,
    name=DATABASE_NAME,
    is_public=False
)

print(db)

ValidationError: 3 validation errors for Database
is_dashboard_enabled
  Field required [type=missing, input_value={'id': 'c77e4bfb-7f16-4a4..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
container
  Field required [type=missing, input_value={'id': 'c77e4bfb-7f16-4a4..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
owner
  Field required [type=missing, input_value={'id': 'c77e4bfb-7f16-4a4..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

The validation error from above can be ignored, everything was created it was only some issue here with parsing.

In [11]:
dbs = client.get_databases()
for db in dbs:
    print(db.id, db.name)

DATABASE_ID = [db.id for db in dbs if db.name == DATABASE_NAME][0]
print("DATABASE_ID:", DATABASE_ID)   

412fb0ce-5299-4d0e-a271-4641b1365b8a data_stewardship_g12_unemployment_prediction
38707917-e942-45c3-a3dd-d2bfc1c106af Vienna Demographic Forecasting
c77e4bfb-7f16-4a47-924b-430778476562 vienna_weather_wet_months
81d82941-cae0-4cba-b27d-1bd883dd713a data_stew_grp22_air_quality
17de844a-62fc-406f-abb2-fffb22b44d01 meine-datenbank
DATABASE_ID: c77e4bfb-7f16-4a47-924b-430778476562


## 4. Create tables in DBRepo and upload data

In [22]:
dir(client)

['__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_upload',
 '_wrapper',
 'analyse_datatypes',
 'analyse_keys',
 'analyse_table_statistics',
 'create_container',
 'create_database',
 'create_database_access',
 'create_identifier',
 'create_subset',
 'create_table',
 'create_table_data',
 'create_view',
 'delete_container',
 'delete_database_access',
 'delete_table',
 'delete_table_data',
 'delete_view',
 'endpoint',
 'get_concepts',
 'get_container',
 'get_containers',
 'get_database',
 'get_database_access',
 'get_databases',
 'get_databases_count',
 'get_identifier',
 'get_identifier_data',
 'get_identifiers',
 'get_image',
 'get_i

In [26]:
df_station = pd.DataFrame([{
    "station_num": 5901,
    "nuts_code": "AT13",
    "district_code": 91900,
    "sub_district_code": 91905,
    "station_name": "Wien - Hohe Warte",
    "latitude_deg": 48.248611,
    "longitude_deg": 16.356944,
    "altitude_m": 202.0
}]).set_index("station_num")

table_station = client.create_table(
    database_id=DATABASE_ID,
    name="station",
    dataframe=df_station,
    is_public=False,
    is_schema_public=False,
    description="Station metadata (Hohe Warte, Vienna). Source: Stadt Wien, CC BY 4.0",
    with_data=False
)

print("station table created:", table_station.id)

NameExistsError: Failed to create table: table name exists

In [30]:
df_time = pd.DataFrame([
    {"time_id": 1, "ref_year": 2020, "ref_month": 1},
    {"time_id": 2, "ref_year": 2020, "ref_month": 2}
]).set_index("time_id")

table_time = client.create_table(
    database_id=DATABASE_ID,
    name="time_dimension",
    dataframe=df_time,
    is_public=False,
    is_schema_public=False,
    description="Time dimension (year, month)",
    with_data=False
)

print(" time_dimension created:", table_time.id)

 time_dimension created: dbfed858-5b3b-4fcd-8f0c-34e000d04bb3


In [31]:
df_weather = pd.DataFrame([
    {
        "measurement_id": 1,
        "station_num": 5901,
        "time_id": 1,
        "t_mean_c": 3.5,
        "p_mean_hpa": 1015.2,
        "precp_sum_mm": 20.1
    }
]).set_index("measurement_id")

table_weather = client.create_table(
    database_id=DATABASE_ID,
    name="weather_measurement",
    dataframe=df_weather,
    is_public=False,
    is_schema_public=False,
    description="Monthly weather measurements (Hohe Warte, Vienna). Source: Stadt Wien, CC BY 4.0",
    with_data=False
)

print("weather_measurement created:", table_weather.id)

✅ weather_measurement created: 125f29b0-c013-4b80-9f66-16a3000dd09e


## 5. Verify – print summary

In [52]:
db = client.get_database(DATABASE_ID)
tables = client.get_tables(DATABASE_ID)

print("=" * 50)
print(f"Database : {db.name}  (id: {db.id})")
print(f"Tables   : {[t.name for t in tables]}")

for t in tables:
    count = client.get_table_data_count(DATABASE_ID, t.id)
    print(f"  {t.name}: {count} rows")

print("=" * 50)
print("T2.1 complete. The DB and Table IDs are:")
print(f"DATABASE_ID             : {DATABASE_ID}")
for t in tables:
    print(t.name, t.id)

Database : vienna_weather_wet_months  (id: c77e4bfb-7f16-4a47-924b-430778476562)
Tables   : ['weather_measurement', 'time_dimension', 'station']
  weather_measurement: 0 rows
  time_dimension: 0 rows
  station: 0 rows
T2.1 complete. The DB and Table IDs are:
DATABASE_ID             : c77e4bfb-7f16-4a47-924b-430778476562
weather_measurement 125f29b0-c013-4b80-9f66-16a3000dd09e
time_dimension dbfed858-5b3b-4fcd-8f0c-34e000d04bb3
station ab680b41-5488-433f-a4d2-df7d8c6854e2


In [53]:
# Delete the incomplete weather_measurement table
client.delete_table(DATABASE_ID, "125f29b0-c013-4b80-9f66-16a3000dd09e")
print("Deleted")

Deleted


In [54]:
dbs = client.get_databases()
for db in dbs:
    print(f"\nDB: {db.name} (id: {db.id})")
    tables = client.get_tables(db.id)
    for t in tables:
        print(f"  Table: {t.name} (id: {t.id})")
        table = client.get_table(db.id, t.id)
        for col in table.columns:
            print(f"    - {col.name}: {col.type}")


DB: Vienna Demographic Forecasting (id: 38707917-e942-45c3-a3dd-d2bfc1c106af)
  Table: Population_statistics (id: 8882a560-da44-4bb9-b14d-28ed91512c4b)
    - population_count: ColumnType.INT
    - nationality_code: ColumnType.VARCHAR
    - district_code: ColumnType.INT
    - year: ColumnType.INT
    - sex_id: ColumnType.INT
    - age_group_id: ColumnType.INT
  Table: Time_dimension (id: b4a40404-3f22-4d6f-a534-8bae624aa576)
    - year: ColumnType.INT
    - reference_date: ColumnType.DATE
  Table: Nationality_group (id: 19a15041-62dc-4676-992a-ea5acdb71ce3)
    - nationality_code: ColumnType.VARCHAR
    - description: ColumnType.VARCHAR
  Table: Age_group (id: a2333bf4-4ce2-42e6-a11d-32f9213b6488)
    - age_group_id: ColumnType.INT
    - age_range: ColumnType.VARCHAR
  Table: Sex (id: c039e55e-e5de-4603-b484-9b2c40f33658)
    - sex_id: ColumnType.INT
    - label: ColumnType.VARCHAR
  Table: District (id: 682a018c-4a31-4324-8f52-40b13139f013)
    - district_code: ColumnType.INT
    - nu

In [57]:
tables = client.get_tables("c77e4bfb-7f16-4a47-924b-430778476562")
for t in tables:
    print(t.name, t.id)

time_dimension dbfed858-5b3b-4fcd-8f0c-34e000d04bb3
station ab680b41-5488-433f-a4d2-df7d8c6854e2


In [12]:
tables = client.get_tables(DATABASE_ID)
for t in tables:
    print(t.name, t.id)

weather_measurement 5ced3fd9-619c-4082-bbe7-0b42b056d028
time_dimension dbfed858-5b3b-4fcd-8f0c-34e000d04bb3
station ab680b41-5488-433f-a4d2-df7d8c6854e2


In [13]:
client.delete_table(DATABASE_ID, "5ced3fd9-619c-4082-bbe7-0b42b056d028")
print("Deleted")

Deleted


In [15]:
import numpy as np
df_weather_full = pd.DataFrame([{
    "measurement_id": 1,
    "station_num": 5901,
    "time_id": 1,
    "t_mean_c": 0.1, "t_max_c": 0.1, "t_min_c": 0.1,
    "mean_t_max_c": 0.1, "mean_t_min_c": 0.1,
    "p_mean_hpa": 0.1, "p_max_hpa": 0.1, "p_min_hpa": 0.1,
    "precp_sum_mm": 0.1,
    "num_precp_01": 5,
    "rel_hum_pct": 0.1, "rel_hum_max_pct": 0.1, "rel_hum_min_pct": 0.1,
    "wind_vel_ms": 0.1, "wind_vel_max_ms": 0.1,
    "num_wind_vel60": 5,
    "sun_h": 0.1,
    "num_clear": 5, "num_cloud": 5,
    "num_frost": 5, "num_ice": 5, "num_summer": 5, "num_heat": 5
}])

int_cols = ["measurement_id", "station_num", "time_id",
            "num_precp_01", "num_wind_vel60", "num_clear",
            "num_cloud", "num_frost", "num_ice", "num_summer", "num_heat"]
for col in int_cols:
    df_weather_full[col] = df_weather_full[col].astype(np.int64)

df_weather_full = df_weather_full.set_index("measurement_id")
print(df_weather_full.dtypes)

station_num          int64
time_id              int64
t_mean_c           float64
t_max_c            float64
t_min_c            float64
mean_t_max_c       float64
mean_t_min_c       float64
p_mean_hpa         float64
p_max_hpa          float64
p_min_hpa          float64
precp_sum_mm       float64
num_precp_01         int64
rel_hum_pct        float64
rel_hum_max_pct    float64
rel_hum_min_pct    float64
wind_vel_ms        float64
wind_vel_max_ms    float64
num_wind_vel60       int64
sun_h              float64
num_clear            int64
num_cloud            int64
num_frost            int64
num_ice              int64
num_summer           int64
num_heat             int64
dtype: object


In [16]:
table_weather = client.create_table(
    database_id=DATABASE_ID,
    name="weather_measurement",
    dataframe=df_weather_full,
    is_public=False,
    is_schema_public=False,
    with_data=False
)
print("Created:", table_weather.id)

Created: 7e7d4391-e837-45f1-91d9-7bcc6a2c1e74
